In [1]:
# ========================================
# ライブラリの import
# ========================================
# TRT系はコメントアウトされている
import argparse

import torch.onnx
from onnxsim import simplify # onnxグラフの簡略化

from mmcv import Config # Configファイルの読み込み
# from mmdeploy.backend.tensorrt.utils import save, search_cuda_version

# mmdetection/mmdet3d系の読み込み
try:
    # If mmdet version > 2.23.0, compat_cfg would be imported and
    # used from mmdet instead of mmdet3d.
    from mmdet.utils import compat_cfg
except ImportError:
    from mmdet3d.utils import compat_cfg

import os
from typing import Dict, Optional, Sequence, Union

import h5py
import mmcv
import numpy as np
import onnx
# import pycuda.driver as cuda
# import tensorrt as trt
import torch
import tqdm
from mmcv.runner import load_checkpoint
# from mmdeploy.apis.core import no_mp
# from mmdeploy.backend.tensorrt.calib_utils import HDF5Calibrator
# from mmdeploy.backend.tensorrt.init_plugins import load_tensorrt_plugin
# from mmdeploy.utils import load_config
from packaging import version
from torch.utils.data import DataLoader

from mmdet3d.datasets import build_dataloader, build_dataset
from mmdet3d.models import build_model
from mmdet.datasets import replace_ImageToTensor
from tools.misc.fuse_conv_bn import fuse_module


import torch.nn as nn
class FastBEV4DExportWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, bev_feat):
        all_cls_scores, all_bbox_preds = self.model(bev_feat)

        return all_cls_scores, all_bbox_preds

class BevEncoderOnly(nn.Module):
    def __init__(self, bev_encoder):
        super().__init__()
        self.bev_encoder = bev_encoder

    def forward(self, bev_feat):
        return self.bev_encoder(bev_feat)

import traceback

args = argparse.Namespace(
    config="../configs/fastbev/paper/fastbev-r50-cbgs-4d-custom.py",
    checkpoint="../ckpts/VADHead/epoch_60.pth",
    work_dir = "./onnx/fastbev4d/",
    prefix="fastbev",
    fp16=False,
    int8=False,
    fuse_conv_bn=False
)

import onnxruntime as ort

/home/kenta/miniconda3/envs/fastbev/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(


# 処理の実行

In [2]:
# onnxファイルの保存先ディレクトリの作成
if not os.path.exists(args.work_dir):
    os.makedirs(args.work_dir)

# TensorRTプラグイン関連の処理
# load_tensorrt_plugin()
# assert 'bev_pool_v2' in get_plugin_names(), \
#     'bev_pool_v2 is not in the plugin list of tensorrt, ' \
#     'please install mmdeploy from ' \
#     'https://github.com/HuangJunJie2017/mmdeploy.git'

if args.int8:
    assert args.fp16

# 出力モデル名の接頭辞を設定
model_prefix = args.prefix
if args.int8:
    model_prefix = model_prefix + '_int8'
elif args.fp16:
    model_prefix = model_prefix + '_fp16'

# configファイルの読み込み
cfg = Config.fromfile(args.config)

# 事前学習済み重みを読み込まない(後で学習済み重みをloadする)
cfg.model.pretrained = None

# モデル名をTRT用に変更
cfg.model.type = "FastBEV4DTRT"

# MMCV系のバージョン互換処理
cfg = compat_cfg(cfg)

# 使用GPUを設定
cfg.gpu_ids = [0]

# データローダーのデフォルト設定
test_dataloader_default_args = dict(samples_per_gpu=1, workers_per_gpu=2, dist=False, shuffle=False)

# データローダーの処理の変更
if isinstance(cfg.data.test, dict):
    cfg.data.test.test_mode = True
    if cfg.data.test_dataloader.get('samples_per_gpu', 1) > 1:
        # Replace 'ImageToTensor' to 'DefaultFormatBundle'
        cfg.data.test.pipeline = replace_ImageToTensor(cfg.data.test.pipeline)
elif isinstance(cfg.data.test, list):
    for ds_cfg in cfg.data.test:
        ds_cfg.test_mode = True
    if cfg.data.test_dataloader.get('samples_per_gpu', 1) > 1:
        for ds_cfg in cfg.data.test:
            ds_cfg.pipeline = replace_ImageToTensor(ds_cfg.pipeline)

# データローダーの作成
test_loader_cfg = {
    **test_dataloader_default_args,
    **cfg.data.get('test_dataloader', {})
}
dataset = build_dataset(cfg.data.test)
data_loader = build_dataloader(dataset, **test_loader_cfg)

# モデルの作成＆重みのロード
cfg.model.train_cfg = None
cfg.model.img_view_transformer.accelerate = True # View変換をTRT用に高速化する
model = build_model(cfg.model, test_cfg=cfg.get('test_cfg'))
load_checkpoint(model, args.checkpoint, map_location='cpu')

# fuse設定
if args.fuse_conv_bn:
    model_prefix = model_prefix + '_fuse'
    model = fuse_module(model)

# 評価モードに設定
model.cuda()
model.eval()

opencv_pp:False


/home/kenta/miniconda3/envs/fastbev/lib/python3.8/site-packages/mmdet/models/backbones/resnet.py:401: UserWarning: DeprecationWarning: pretrained is deprecated, please use "init_cfg" instead
  warnings.warn('DeprecationWarning: pretrained is deprecated, '


load checkpoint from local path: ../ckpts/VADHead/epoch_60.pth


FastBEV4DTRT(
  (pts_bbox_head): VADHead(
    (loss_cls): FocalLoss()
    (loss_bbox): L1Loss()
    (loss_iou): GIoULoss()
    (activate): ReLU(inplace=True)
    (positional_encoding): SinePositionalEncoding(num_feats=128, temperature=10000, normalize=True, scale=6.283185307179586, eps=1e-06)
    (transformer): VADPerceptionTransformer(
      (decoder): DetectionTransformerDecoder(
        (layers): ModuleList(
          (0-2): 3 x DetrTransformerDecoderLayer(
            (attentions): ModuleList(
              (0): MultiheadAttention(
                (attn): MultiheadAttention(
                  (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
                )
                (proj_drop): Dropout(p=0.0, inplace=False)
                (dropout_layer): Dropout(p=0.1, inplace=False)
              )
              (1): CustomMSDeformableAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (sampling_offsets): Linear(in

# ONNXモデルのエクスポート

In [3]:
# import ipdb; ipdb.set_trace()
for i, data in enumerate(data_loader):
    if i == 0:
        continue
    
    # 入力データを過去と現在フレームに分ける(元のソフト)
    inputs = [d.cuda() for d in data['img_inputs'][0]]
    imgs, sensor2keyegos, ego2globals, intrins, post_rots, post_trans, bda, _ = model.prepare_inputs(inputs)
    
    img_curr, img_prev = imgs
    sensor2keyegos_curr, sensor2keyegos_prev = sensor2keyegos
    ego2globals_curr, ego2globals_prev = ego2globals
    intrins_curr, intrins_prev = intrins
    post_rots_curr, post_rots_prev = post_rots
    post_trans_curr, post_trans_prev = post_trans

    # BEV特徴量の作成
    # 現在フレーム
    inputs_curr = (img_curr, sensor2keyegos_curr, ego2globals_curr, intrins_curr, post_rots_curr, post_trans_curr, bda, None)
    bev_feat_curr, _ = model.prepare_bev_feat(*inputs_curr)

    # 過去フレーム
    inputs_prev = (img_prev, sensor2keyegos_curr, ego2globals_curr, intrins_prev, post_rots_prev, post_trans_prev, bda, None)
    bev_feat_prev, _ = model.prepare_bev_feat(*inputs_prev)

    # BEVの結合
    bev_feat_list = [bev_feat_curr, bev_feat_prev]
    bev_feat_list[1] = model.shift_feature(bev_feat_list[1], [sensor2keyegos_curr, sensor2keyegos_prev], bda)
    bev_feats = torch.cat(bev_feat_list, dim=1)

    #wrapper = FastBEV4DExportWrapper(model).cuda().eval()
    #wrapper_bev = BevEncoderOnly(model.bev_encoder).cuda().eval()
    #bev_feats = bev_feats.detach().float().contiguous()

    #print("bev_feat_curr.shape =", bev_feat_curr.shape)
    #print("bev_feat_prev.shape =", bev_feat_prev.shape)
    #print("bev_feats.shape     =", bev_feats.shape)

    with torch.no_grad():
        y = model(bev_feats)
        #print("wrapper_bev output shape =", y.shape)

    break

/home/kenta/fastbev/mmdet3d/datasets/pipelines/loading.py:1307: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1682343998658/work/torch/csrc/utils/tensor_new.cpp:245.)
  gt_boxes, gt_labels = torch.Tensor(gt_boxes), torch.tensor(gt_labels)
/home/kenta/fastbev/mmdet3d/datasets/pipelines/loading.py:1307: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1682343998658/work/torch/csrc/utils/tensor_new.cpp:245.)
  gt_boxes, gt_labels = torch.Tensor(gt_boxes), torch.tensor(gt_labels)
/home/kenta/fastbev/mmdet3d/datasets/pipelines/loading.py:1307: UserWarning: Creating a tensor from a list of numpy.ndarra

use_depth True
img_coord:  [tensor([3081, 3037, 3037,  ...,    0,    0,    0], device='cuda:0')]
use_depth True


In [6]:
bev_feats.shape

torch.Size([1, 160, 128, 128])

In [4]:
# ONNX export(FastBEVTRT)
with torch.no_grad():
    torch.onnx.export(
        model,
        (
            bev_feats.contiguous(),  # [B, 160, H, W] 現在BEV + 過去BEVをチャネル方向に結合した特徴
        ),
        args.work_dir + model_prefix + '_4d.onnx', # onnxファイルの保存先
        opset_version=16, # 16でないと、F.sampleに非対応
        input_names=["bev_feats"],
        output_names=["all_cls_scores", "all_bbox_preds"],
        do_constant_folding=False,
        training=torch.onnx.TrainingMode.EVAL
    )

/home/kenta/fastbev/mmdet3d/models/VAD/VAD_transformer.py:198: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  spatial_shapes=torch.tensor([[bev_h, bev_w]], device=query.device),
/home/kenta/fastbev/mmdet3d/models/VAD/VAD_transformer.py:199: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  level_start_index=torch.tensor([0], device=query.device),
/home/kenta/fastbev/mmdet3d/models/VAD/modules/decoder.py:287: TracerWarning: Converting a tensor to a Python boolean might cause the trac

================ Diagnostic Run torch.onnx.export version 2.0.1 ================
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



In [5]:
import onnxruntime as ort
fastbev4d = ort.InferenceSession(
    "./onnx/fastbev4d/fastbev_4d.onnx",
    #providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    providers=["CPUExecutionProvider"]
)

print("\n===fastbev4d===")
print("=== Inputs ===")
for i, x in enumerate(fastbev4d.get_inputs()):
    print(f"[{i}]")
    print(" name :", x.name)
    print(" shape:", x.shape)
    print(" type :", x.type)

print("\n=== Outputs ===")
for i, x in enumerate(fastbev4d.get_outputs()):
    print(f"[{i}]")
    print(" name :", x.name)
    print(" shape:", x.shape)
    print(" type :", x.type)


===fastbev4d===
=== Inputs ===
[0]
 name : bev_feats
 shape: [1, 160, 128, 128]
 type : tensor(float)

=== Outputs ===
[0]
 name : all_cls_scores
 shape: [1, 300, 10]
 type : tensor(float)
[1]
 name : all_bbox_preds
 shape: [1, 300, 10]
 type : tensor(float)


In [6]:
inp = bev_feats.detach().contiguous().float().cpu().numpy()
zero = np.zeros_like(inp)
rand = np.random.randn(*inp.shape).astype(np.float32)

outs_real = fastbev4d.run(None, {"bev_feats": inp})
outs_zero = fastbev4d.run(None, {"bev_feats": zero})
outs_rand = fastbev4d.run(None, {"bev_feats": rand})

print("cls real-zero diff:", np.abs(outs_real[0] - outs_zero[0]).max())
print("bbox real-zero diff:", np.abs(outs_real[1] - outs_zero[1]).max())
print("cls real-rand diff:", np.abs(outs_real[0] - outs_rand[0]).max())
print("bbox real-rand diff:", np.abs(outs_real[1] - outs_rand[1]).max())

cls real-zero diff: 7.560767
bbox real-zero diff: 85.176605
cls real-rand diff: 6.945304
bbox real-rand diff: 86.02507


In [7]:
# 物体認識の実行
outs = fastbev4d.run(
    None,
    {"bev_feats": bev_feats.detach().cpu().numpy()}
)

In [8]:
y[1]

tensor([[[-1.0582e+01,  1.6629e+01, -3.5616e-01,  ..., -8.5421e-02,
          -2.4181e-06, -4.9061e-06],
         [ 2.9419e+01,  1.7579e+01, -3.0160e-01,  ...,  5.5703e-01,
           1.8400e-01, -2.7623e-02],
         [ 1.4205e+01,  9.1903e+00, -5.2790e-01,  ...,  9.9266e-01,
           8.7465e-01, -1.1111e-02],
         ...,
         [ 3.9806e+01, -2.3520e+01,  1.5047e+00,  ..., -9.5260e-02,
           7.7717e-03, -5.3144e-01],
         [ 3.0214e+01, -1.7664e+01,  1.5616e+00,  ..., -1.1011e-01,
           7.6174e-08, -4.8446e-07],
         [ 2.8684e+01,  1.5476e+01, -3.2082e-01,  ...,  6.0113e-01,
          -2.9922e-08, -5.5332e-06]]], device='cuda:0')

In [9]:
outs[1]

array([[[-1.0587299e+01,  1.6629772e+01, -3.5626489e-01, ...,
         -8.2448468e-02, -2.4156352e-06, -4.9082178e-06],
        [ 2.9419041e+01,  1.7579250e+01, -3.0162153e-01, ...,
          5.5713159e-01,  1.8419661e-01, -2.7673958e-02],
        [ 1.4205326e+01,  9.1904106e+00, -5.2754855e-01, ...,
          9.9266803e-01,  8.7484729e-01, -1.1113353e-02],
        ...,
        [ 3.9806950e+01, -2.3520594e+01,  1.5047828e+00, ...,
         -9.5267460e-02,  7.7914423e-03, -5.3181303e-01],
        [ 3.0213345e+01, -1.7663452e+01,  1.5616062e+00, ...,
         -1.1006923e-01,  7.6007524e-08, -4.8442058e-07],
        [ 2.8684162e+01,  1.5475582e+01, -3.2082599e-01, ...,
          6.0112143e-01, -3.0090831e-08, -5.5327178e-06]]], dtype=float32)

In [10]:
model.eval()

bev_input = bev_feats.detach().contiguous().float()

with torch.no_grad():
    y = model(bev_input)

torch_cls = y[0].detach().cpu().numpy()
torch_bbox = y[1].detach().cpu().numpy()

onnx_outs = fastbev4d.run(
    None,
    {"bev_feats": bev_input.cpu().numpy().astype(np.float32)}
)

onnx_cls = onnx_outs[0]
onnx_bbox = onnx_outs[1]

print("cls max diff :", np.abs(torch_cls - onnx_cls).max())
print("cls mean diff:", np.abs(torch_cls - onnx_cls).mean())

print("bbox max diff :", np.abs(torch_bbox - onnx_bbox).max())
print("bbox mean diff:", np.abs(torch_bbox - onnx_bbox).mean())

cls max diff : 0.014299393
cls mean diff: 0.000478463
bbox max diff : 0.044216156
bbox mean diff: 0.000460356


In [11]:
# PyTorch出力
torch_cls = y[0].detach().cpu()
torch_bbox = y[1].detach().cpu()

# ONNX出力
onnx_cls = torch.from_numpy(onnx_outs[0])
onnx_bbox = torch.from_numpy(onnx_outs[1])

# raw出力のscore差
torch_score = torch_cls.sigmoid()
onnx_score = onnx_cls.sigmoid()

print("score max diff :", (torch_score - onnx_score).abs().max().item())
print("score mean diff:", (torch_score - onnx_score).abs().mean().item())

score max diff : 0.00034934282302856445
score mean diff: 3.509077259877813e-06


In [4]:
x = model.bev_encoder(bev_feats.detach().contiguous().float())

In [6]:
import torch

class HeadOnlyTRT(torch.nn.Module):
    def __init__(self, pts_bbox_head):
        super().__init__()
        self.pts_bbox_head = pts_bbox_head

    def forward(self, x):
        bbox_pts = self.pts_bbox_head([x])

        cls_out = bbox_pts["all_cls_scores"][-1]
        bbox_out = bbox_pts["all_bbox_preds"][-1]

        return x, cls_out, bbox_out

In [7]:
head_model = HeadOnlyTRT(model.pts_bbox_head).eval()

with torch.no_grad():
    torch.onnx.export(
        head_model,
        (x.detach().contiguous().float(),),
        args.work_dir + model_prefix + '_head.onnx',
        opset_version=11,
        input_names=["x"],
        output_names=["x_out", "all_cls_scores", "all_bbox_preds"],
        do_constant_folding=False,
        training=torch.onnx.TrainingMode.EVAL
    )

/home/kenta/fastbev/mmdet3d/models/VAD/VAD_transformer.py:198: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  spatial_shapes=torch.tensor([[bev_h, bev_w]], device=query.device),
/home/kenta/fastbev/mmdet3d/models/VAD/VAD_transformer.py:199: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  level_start_index=torch.tensor([0], device=query.device),
/home/kenta/fastbev/mmdet3d/models/VAD/modules/decoder.py:287: TracerWarning: Converting a tensor to a Python boolean might cause the trac

================ Diagnostic Run torch.onnx.export version 2.0.1 ================
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



In [8]:
import onnxruntime as ort

head_onnx = ort.InferenceSession(
    args.work_dir + model_prefix + '_head.onnx',
    providers=["CPUExecutionProvider"]
)

print("=== Inputs ===")
for i, inp in enumerate(head_onnx.get_inputs()):
    print(f"[{i}]")
    print(" name :", inp.name)
    print(" shape:", inp.shape)
    print(" type :", inp.type)

print("=== Outputs ===")
for i, out in enumerate(head_onnx.get_outputs()):
    print(f"[{i}]")
    print(" name :", out.name)
    print(" shape:", out.shape)
    print(" type :", out.type)

=== Inputs ===
[0]
 name : x
 shape: [1, 256, 128, 128]
 type : tensor(float)
=== Outputs ===
[0]
 name : x_out
 shape: [1, 256, 128, 128]
 type : tensor(float)
[1]
 name : all_cls_scores
 shape: [1, 300, 10]
 type : tensor(float)
[2]
 name : all_bbox_preds
 shape: [1, 300, 10]
 type : tensor(float)


In [9]:
inp = x.detach().contiguous().float().cpu().numpy()
zero = np.zeros_like(inp)

outs_real = head_onnx.run(None, {"x": inp})
outs_zero = head_onnx.run(None, {"x": zero})

print("x_out diff:", np.abs(outs_real[0] - outs_zero[0]).max())
print("cls diff:", np.abs(outs_real[1] - outs_zero[1]).max())
print("bbox diff:", np.abs(outs_real[2] - outs_zero[2]).max())

x_out diff: 2.9427288
cls diff: 0.0
bbox diff: 0.0


In [11]:
head_model = HeadOnlyTRT(model.pts_bbox_head).eval()

with torch.no_grad():
    y_real = head_model(x)[1]
    y_zero = head_model(torch.zeros_like(x))[1]

print("head pytorch real-zero max diff:", (y_real - y_zero).abs().max().item())

head pytorch real-zero max diff: 78.72219848632812


In [5]:
import torch
import torch.nn as nn

class VADTransformerOnlyTRT(nn.Module):
    def __init__(self, pts_bbox_head):
        super().__init__()
        self.transformer = pts_bbox_head.transformer
        self.query_embedding = pts_bbox_head.query_embedding

        self.bev_h = pts_bbox_head.bev_h
        self.bev_w = pts_bbox_head.bev_w

        self.reg_branches = pts_bbox_head.reg_branches
        self.cls_branches = pts_bbox_head.cls_branches

        self.with_box_refine = pts_bbox_head.with_box_refine
        self.as_two_stage = pts_bbox_head.as_two_stage

    def forward(self, x):
        """
        Args:
            x: BEV encoder後の特徴 [B, C, H, W]
        Returns:
            hs_last: 最終decoder層のquery特徴
            reference_last: 最終decoder層で使う参照点
        """
        dtype = x.dtype
        object_query_embeds = self.query_embedding.weight.to(dtype)

        outputs = self.transformer(
            x,
            object_query_embeds,
            self.bev_h,
            self.bev_w,
            reg_branches=self.reg_branches if self.with_box_refine else None,
            cls_branches=self.cls_branches if self.as_two_stage else None,
        )

        bev_embed, hs, init_reference, inter_references = outputs

        # 元のVADHead.forwardと同じ処理を維持
        hs = hs.permute(0, 2, 1, 3)

        # 最終層だけ使う
        last_lvl = hs.shape[0] - 1
        hs_last = hs[last_lvl]

        # 元の処理では lvl==0 のとき init_reference、
        # lvl>0 のとき inter_references[lvl - 1] を使う
        if last_lvl == 0:
            reference_last = init_reference
        else:
            reference_last = inter_references[last_lvl - 1]

        return hs_last, reference_last

In [9]:
transformer_model = VADTransformerOnlyTRT(model.pts_bbox_head).eval()

x_export = x.detach().contiguous().float()

with torch.no_grad():
    torch.onnx.export(
        transformer_model,
        (x_export,),
        args.work_dir + model_prefix + "_transformer.onnx",
        opset_version=16,
        input_names=["x"],
        output_names=["hs_last", "reference_last"],
        do_constant_folding=False,
        training=torch.onnx.TrainingMode.EVAL
    )

/tmp/ipykernel_181298/2606002611.py:50: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if last_lvl == 0:


================ Diagnostic Run torch.onnx.export version 2.0.1 ================
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



In [7]:
transformer_onnx = ort.InferenceSession(
    args.work_dir + model_prefix + "_transformer.onnx",
    providers=["CPUExecutionProvider"]
)

print("=== Inputs ===")
for i in transformer_onnx.get_inputs():
    print(i.name, i.shape, i.type)

print("=== Outputs ===")
for o in transformer_onnx.get_outputs():
    print(o.name, o.shape, o.type)

NameError: name 'ort' is not defined

In [14]:
transformer_model = VADTransformerOnlyTRT(model.pts_bbox_head).eval().cpu()
x_cpu = x.detach().contiguous().float().cpu()

onnx_path_transformer = args.work_dir + model_prefix + "_transformer_cpu.onnx"

with torch.no_grad():
    torch.onnx.export(
        transformer_model,
        (x_cpu,),
        onnx_path_transformer,
        opset_version=16,
        input_names=["x"],
        output_names=["hs_last", "reference_last"],
        do_constant_folding=False,
        training=torch.onnx.TrainingMode.EVAL
    )

/home/kenta/miniconda3/envs/fastbev/lib/python3.8/site-packages/mmcv/ops/multi_scale_deform_attn.py:124: TracerWarning: Iterating over a tensor might cause the trace to be incorrect. Passing a tensor of different shape won't change the number of iterations executed (and might lead to errors or silently give incorrect results).
  value_list = value.split([H_ * W_ for H_, W_ in value_spatial_shapes],
/home/kenta/miniconda3/envs/fastbev/lib/python3.8/site-packages/mmcv/ops/multi_scale_deform_attn.py:128: TracerWarning: Iterating over a tensor might cause the trace to be incorrect. Passing a tensor of different shape won't change the number of iterations executed (and might lead to errors or silently give incorrect results).
  for level, (H_, W_) in enumerate(value_spatial_shapes):
/tmp/ipykernel_120219/2606002611.py:50: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated

================ Diagnostic Run torch.onnx.export version 2.0.1 ================
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



In [11]:
transformer_onnx = ort.InferenceSession(
    args.work_dir + model_prefix + "_transformer.onnx",
    providers=["CPUExecutionProvider"]
)

print("=== Inputs ===")
for i in transformer_onnx.get_inputs():
    print(i.name, i.shape, i.type)

print("=== Outputs ===")
for o in transformer_onnx.get_outputs():
    print(o.name, o.shape, o.type)

=== Inputs ===
x [1, 256, 128, 128] tensor(float)
=== Outputs ===
hs_last [1, 300, 256] tensor(float)
reference_last [1, 300, 3] tensor(float)


In [12]:
inp = x.detach().contiguous().float().cpu().numpy()
zero = np.zeros_like(inp)
rand = np.random.randn(*inp.shape).astype(np.float32)

outs_real = transformer_onnx.run(None, {"x": inp})
outs_zero = transformer_onnx.run(None, {"x": zero})
outs_rand = transformer_onnx.run(None, {"x": rand})

print("hs_last real-zero diff:",
      np.abs(outs_real[0] - outs_zero[0]).max())

print("reference_last real-zero diff:",
      np.abs(outs_real[1] - outs_zero[1]).max())

print("hs_last real-rand diff:",
      np.abs(outs_real[0] - outs_rand[0]).max())

print("reference_last real-rand diff:",
      np.abs(outs_real[1] - outs_rand[1]).max())

hs_last real-zero diff: 0.8747932
reference_last real-zero diff: 0.7595157
hs_last real-rand diff: 1.1162064
reference_last real-rand diff: 0.75707996


In [10]:
print("=== Inputs ===")
for i, inp_meta in enumerate(transformer_onnx.get_inputs()):
    print(f"[{i}]")
    print(" name :", inp_meta.name)
    print(" shape:", inp_meta.shape)
    print(" type :", inp_meta.type)

print("=== Outputs ===")
for i, out_meta in enumerate(transformer_onnx.get_outputs()):
    print(f"[{i}]")
    print(" name :", out_meta.name)
    print(" shape:", out_meta.shape)
    print(" type :", out_meta.type)

=== Inputs ===


NameError: name 'transformer_onnx' is not defined

In [12]:
import onnxruntime as ort
import numpy as np

transformer_onnx = ort.InferenceSession(
    onnx_path_transformer,
    providers=["CPUExecutionProvider"]
)

inp = x.detach().cpu().numpy().astype(np.float32)
zero = np.zeros_like(inp)
rand = np.random.randn(*inp.shape).astype(np.float32)

outs_real = transformer_onnx.run(None, {"x": inp})
outs_zero = transformer_onnx.run(None, {"x": zero})
outs_rand = transformer_onnx.run(None, {"x": rand})

print("hs_last real-zero diff:",
      np.abs(outs_real[0] - outs_zero[0]).max())
print("reference_last real-zero diff:",
      np.abs(outs_real[1] - outs_zero[1]).max())

print("hs_last real-rand diff:",
      np.abs(outs_real[0] - outs_rand[0]).max())
print("reference_last real-rand diff:",
      np.abs(outs_real[1] - outs_rand[1]).max())

InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Invalid input name: x

In [16]:
bev_feats.shape

torch.Size([1, 160, 128, 128])

In [19]:
# 物体認識の実行
outs = fastbev4d.run(
    None,
    {"bev_feats": bev_feats.detach().cpu().numpy()}
)

In [23]:
y[1]

tensor([[[-1.0582e+01,  1.6629e+01, -3.5616e-01,  ..., -8.5421e-02,
          -2.4181e-06, -4.9061e-06],
         [ 2.9419e+01,  1.7579e+01, -3.0160e-01,  ...,  5.5703e-01,
           1.8400e-01, -2.7623e-02],
         [ 1.4205e+01,  9.1903e+00, -5.2790e-01,  ...,  9.9266e-01,
           8.7465e-01, -1.1111e-02],
         ...,
         [ 3.9806e+01, -2.3520e+01,  1.5047e+00,  ..., -9.5260e-02,
           7.7717e-03, -5.3144e-01],
         [ 3.0214e+01, -1.7664e+01,  1.5616e+00,  ..., -1.1011e-01,
           7.6174e-08, -4.8446e-07],
         [ 2.8684e+01,  1.5476e+01, -3.2082e-01,  ...,  6.0113e-01,
          -2.9922e-08, -5.5332e-06]]], device='cuda:0')

In [26]:
outs[1]

array([[[ 3.9673367e+01,  2.9777843e+01,  1.5746284e+00, ...,
          1.5688187e-02,  1.9920863e-06,  1.2301340e-06],
        [ 3.2806271e+01,  2.9669888e+01,  1.5688189e+00, ...,
          1.4186649e-02,  2.0054624e-06,  1.2830541e-06],
        [ 1.2300121e+01,  1.9889221e+00,  1.4557300e+00, ...,
         -2.5890838e-02,  2.1687979e-06,  1.5888326e-06],
        ...,
        [ 4.1537342e+01, -2.6698105e+01,  1.5776374e+00, ...,
         -1.0652000e-01,  1.6994019e-06,  1.0626101e-06],
        [ 3.1669544e+01, -2.4219822e+01,  1.5530559e+00, ...,
         -2.8546649e-01,  1.4625407e-06,  1.0362070e-06],
        [ 3.2897350e+01,  2.2706577e+01,  1.5527482e+00, ...,
          1.7004533e-02,  2.1398182e-06,  1.2531999e-06]]], dtype=float32)

In [10]:
# ONNX export(FastBEVTRT)
with torch.no_grad():
    torch.onnx.export(
        model,
        (
            bev_feats.contiguous(),  # [B, 160, H, W] 現在BEV + 過去BEVをチャネル方向に結合した特徴
        ),
        args.work_dir + model_prefix + '_4d.onnx', # onnxファイルの保存先
        opset_version=11, # 16でないと、F.sampleに非対応
        input_names=["bev_feats"],
        output_names=["all_cls_scores", "all_bbox_preds"]
    )

================ Diagnostic Run torch.onnx.export version 2.0.1 ================
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



In [5]:
# ONNXモデルの検証(FastBEV4DTRT)
onnx_model_4d = onnx.load(args.work_dir + model_prefix + '_4d.onnx')
try:
    onnx.checker.check_model(onnx_model_4d)
except Exception:
    print('ONNX Model(FastBEV4DTRT) Incorrect')
else:
    print('ONNX Model(FastBEV4DTRT) Correct')

ONNX Model(FastBEV4DTRT) Correct


In [6]:
import onnxruntime as ort
fastbev4d = ort.InferenceSession(
    "./onnx/fastbev4d/fastbev_4d.onnx",
    #providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    providers=["CPUExecutionProvider"]
)

print("\n===fastbev4d===")
print("=== Inputs ===")
for i, x in enumerate(fastbev4d.get_inputs()):
    print(f"[{i}]")
    print(" name :", x.name)
    print(" shape:", x.shape)
    print(" type :", x.type)

print("\n=== Outputs ===")
for i, x in enumerate(fastbev4d.get_outputs()):
    print(f"[{i}]")
    print(" name :", x.name)
    print(" shape:", x.shape)
    print(" type :", x.type)

In [7]:
print("\n===fastbev4d===")
print("=== Inputs ===")
for i, x in enumerate(fastbev4d.get_inputs()):
    print(f"[{i}]")
    print(" name :", x.name)
    print(" shape:", x.shape)
    print(" type :", x.type)

print("\n=== Outputs ===")
for i, x in enumerate(fastbev4d.get_outputs()):
    print(f"[{i}]")
    print(" name :", x.name)
    print(" shape:", x.shape)
    print(" type :", x.type)


===fastbev4d===
=== Inputs ===

=== Outputs ===
[0]
 name : all_cls_scores
 shape: [1, 300, 10]
 type : tensor(float)
[1]
 name : all_bbox_preds
 shape: [1, 300, 10]
 type : tensor(float)


In [9]:
# 計算グラフの簡素化
onnx_simp_4d, check_4d = simplify(onnx_model_4d)
assert check_4d, "Simplified ONNX model could not be validated"

RuntimeError: /project/third_party/onnx-optimizer/third_party/onnx/onnx/common/ir.h:670: insertAfter: Assertion `!inGraphList() && n->inGraphList()` failed.

In [6]:
y[1]

tensor([[[-1.0582e+01,  1.6629e+01, -3.5616e-01,  ..., -8.5421e-02,
          -2.4181e-06, -4.9061e-06],
         [ 2.9419e+01,  1.7579e+01, -3.0160e-01,  ...,  5.5703e-01,
           1.8400e-01, -2.7623e-02],
         [ 1.4205e+01,  9.1903e+00, -5.2790e-01,  ...,  9.9266e-01,
           8.7465e-01, -1.1111e-02],
         ...,
         [ 3.9806e+01, -2.3520e+01,  1.5047e+00,  ..., -9.5260e-02,
           7.7717e-03, -5.3144e-01],
         [ 3.0214e+01, -1.7664e+01,  1.5616e+00,  ..., -1.1011e-01,
           7.6174e-08, -4.8446e-07],
         [ 2.8684e+01,  1.5476e+01, -3.2082e-01,  ...,  6.0113e-01,
          -2.9922e-08, -5.5332e-06]]], device='cuda:0')

In [27]:
inp = bev_feats.detach().contiguous().float().cpu().numpy()

outs_real = fastbev4d.run(None, {"bev_feats": inp})[1]
outs_zero = fastbev4d.run(None, {"bev_feats": np.zeros_like(inp)})[1]
outs_rand = fastbev4d.run(None, {"bev_feats": np.random.randn(*inp.shape).astype(np.float32)})[1]

print("real-zero max diff:", np.abs(outs_real - outs_zero).max())
print("real-rand max diff:", np.abs(outs_real - outs_rand).max())

real-zero max diff: 0.0
real-rand max diff: 0.0


In [28]:
model.eval()

with torch.no_grad():
    y_real = model(bev_feats.detach().contiguous().float())[1]
    y_zero = model(torch.zeros_like(bev_feats.detach().contiguous().float()))[1]

print("torch real-zero max diff:", (y_real - y_zero).abs().max().item())

torch real-zero max diff: 85.17413330078125
